In [ ]:
# =============================================================================
# jobs/25_warning_history_periods.py
# When was each flood alert, flood warning and severe flood warning in force, and for
# how long? Built from a file of every message the EA has issued.
#
# What this script does:
#   1. Reads the message history (a parquet file, or a folder of them, in a Databricks
#      volume) with one row per message: area code (TACode), time (dateTime) and
#      severity ("Flood alert", "Flood warning", "Severe flood warning" or "Warning no
#      longer in force").
#   2. Replays each area's messages in time order with the rules in
#      utils/warning_history.py:
#        Flood alert                  switches an alert on.
#        Flood warning                switches a warning on, or turns a severe flood
#                                     warning down one level.
#        Severe flood warning         raises an existing warning to severe.
#        Warning no longer in force   switches the area off, from any level.
#      A message at the level an area is already at is an update, not a change.
#   3. Writes two tables, each with start, end and duration:
#        ea_warning_episodes  one row per run in force at ANY level. A warning that goes
#                             severe and comes back down is one episode.
#        ea_warning_periods   one row per run at ONE level inside an episode, so that
#                             warning has a flood warning period, a severe period and a
#                             flood warning period again.
#   4. Overwrites both tables in full: the file is the whole record, so a re-run gives
#      the same answer. It never changes anything outside these two tables.
#
# TIMES. The times are used as written in the file (dateTime, which carries a "Z"). If the
# source recorded UK clock time and labelled it Z, a duration that spans a clock change
# (late March or late October) is out by an hour. Durations inside one day are not affected.
#
# OPEN EPISODES. Anything with no "no longer in force" by the end of the file is open: it
# has no end_time, its duration is measured to the latest message in the file, and
# stale_open is true when that is over 30 days (a "no longer in force" is probably missing).
#
# Parameter (widget):
#   source_path   the parquet file or folder, for example
#                 /Volumes/<catalog>/<schema>/<volume>/2026_06_08_All_Messages_FWIS.parquet
#
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType
)
from config import TBL_WARNING_EPISODES, TBL_WARNING_PERIODS
from utils.helpers import get_spark, utc_now
from utils.warning_history import normalise_messages, build_history

dbutils.widgets.text("source_path", "", "Parquet file or folder in a volume")
source_path = dbutils.widgets.get("source_path").strip()
if not source_path:
    raise Exception(
        "Set the source_path widget to the parquet file (or folder) holding the message history, "
        "for example /Volumes/<catalog>/<schema>/<volume>/<file>.parquet"
    )

spark   = get_spark()
now_iso = utc_now().isoformat()

In [ ]:
# =============================================================================
# READ THE MESSAGES AND CHECK THEM
# pandas (not Spark) reads the file, because it holds two columns called "severity" and
# "Severity": Spark treats those as the same name, pandas does not.
# =============================================================================

raw = pd.read_parquet(source_path)
print(f"Read {len(raw):,} messages from {source_path}")
print(f"Columns: {list(raw.columns)}")

events, report = normalise_messages(raw)
print(f"\nUsable messages: {len(events):,} for {events['ea_area_code'].nunique():,} areas, "
      f"{events['time'].min()} to {events['time'].max()}")
print(f"  no area code: {report['no_code']}   no readable time: {report['no_time']}   "
      f"unrecognised severity: {report['unknown_severity']}   exact repeats dropped: {report['duplicates']}")
if report["unknown_values"]:
    print(f"  severity values not recognised: {report['unknown_values']}")
if report["disagreeing_levels"]:
    print(f"  WARNING: {report['disagreeing_levels']} rows where the text and the numeric severity disagree "
          f"(the text was used).")
print("\nMessages by severity:")
print(events["level"].value_counts().to_string())

In [ ]:
# =============================================================================
# BUILD THE EPISODES AND PERIODS, AND WRITE THEM
# =============================================================================

episodes, periods, stats = build_history(events)
as_of = events["time"].max()

print(f"\nEpisodes: {len(episodes):,}   periods: {len(periods):,}   "
      f"'no longer in force' with nothing on: {stats['orphan_off']}")
print("\nEpisodes by area type, with the median and longest duration in hours:")
closed = episodes[~episodes["is_open"]]
print(closed.groupby("area_type")["duration_hours"].agg(["count", "median", "max"]).round(1).to_string())
n_open = int(episodes["is_open"].sum())
n_stale = int(episodes["stale_open"].sum())
print(f"\nStill in force at the end of the file: {n_open}   (open for over 30 days, probably missing their end: {n_stale})")
print(f"Episodes that went severe: {int(episodes['had_severe'].sum())}")

EPISODE_SCHEMA = StructType([
    StructField("episode_id",       StringType(),    True),
    StructField("ea_area_code",     StringType(),    True),
    StructField("area_type",        StringType(),    True),   # flood_alert or flood_warning, from the area code
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),   # null while open
    StructField("duration_minutes", DoubleType(),    True),   # to the end of the file while open
    StructField("duration_hours",   DoubleType(),    True),
    StructField("start_level",      StringType(),    True),
    StructField("peak_level",       StringType(),    True),
    StructField("had_severe",       BooleanType(),   True),
    StructField("severe_minutes",   DoubleType(),    True),   # time at severe inside the episode
    StructField("n_severe_periods", IntegerType(),   True),
    StructField("n_messages",       IntegerType(),   True),   # messages while in force, updates included
    StructField("ended_by",         StringType(),    True),   # no_longer_in_force or open
    StructField("is_open",          BooleanType(),   True),
    StructField("stale_open",       BooleanType(),   True),
    StructField("data_as_of",       TimestampType(), True),   # the latest message in the file
    StructField("loaded_at",        StringType(),    True),
])

PERIOD_SCHEMA = StructType([
    StructField("period_id",        StringType(),    True),
    StructField("episode_id",       StringType(),    True),
    StructField("ea_area_code",     StringType(),    True),
    StructField("level",            StringType(),    True),   # flood_alert, flood_warning or severe_flood_warning
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),
    StructField("duration_minutes", DoubleType(),    True),
    StructField("duration_hours",   DoubleType(),    True),
    StructField("started_by",       StringType(),    True),   # issued, upgraded or downgraded
    StructField("ended_by",         StringType(),    True),   # upgraded, downgraded, no_longer_in_force or open
    StructField("is_open",          BooleanType(),   True),
    StructField("data_as_of",       TimestampType(), True),
    StructField("loaded_at",        StringType(),    True),
])


def to_rows(frame, schema):
    """Rows as tuples in schema order, with pandas NaT and NaN as None."""
    frame = frame.copy()
    frame["data_as_of"] = as_of
    frame["loaded_at"] = now_iso
    rows = []
    for rec in frame[[f.name for f in schema.fields]].itertuples(index=False, name=None):
        row = []
        for value, field in zip(rec, schema.fields):
            if value is None or (not isinstance(value, (bool, str)) and pd.isna(value)):
                row.append(None)
            elif isinstance(field.dataType, TimestampType):
                row.append(pd.Timestamp(value).to_pydatetime())
            elif isinstance(field.dataType, BooleanType):
                row.append(bool(value))
            elif isinstance(field.dataType, IntegerType):
                row.append(int(value))
            elif isinstance(field.dataType, DoubleType):
                row.append(float(value))
            else:
                row.append(str(value))
        rows.append(tuple(row))
    return rows


for table, frame, schema, comment in (
    (TBL_WARNING_EPISODES, episodes, EPISODE_SCHEMA,
     "Each run of a flood alert, flood warning or severe flood warning being in force in an area, "
     "from the message that switched it on to the Warning no longer in force that ended it, with start, "
     "end and duration. A warning that goes severe and back is one episode. Built by 25_warning_history_periods "
     "from the EA message history. Times are as written in the source. is_open means no end by the end of the file."),
    (TBL_WARNING_PERIODS, periods, PERIOD_SCHEMA,
     "Each run at ONE level (flood alert, flood warning or severe flood warning) inside an episode in "
     "ea_warning_episodes, with start, end and duration. Built by 25_warning_history_periods."),
):
    (
        spark.createDataFrame(to_rows(frame, schema), schema).write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")
    print(f"Wrote {spark.table(table).count():,} rows to {table}")

dbutils.notebook.exit("success")